# SymbioPan v8 — CellPath Training

**Architecture**: Virchow2 ViT-H/14 encoder → ConvNeXt-Tiny CNN backbone → HierarchicalFPN → ParallelDecoders (DeepLabV3+ tissue, CellViT++ nuclei, HoVerNeXt NP/HV, BoundaryAttention)

**No data preprocessing or model pretraining steps** — assumes `dataset_processed/` is ready.

In [ ]:
import sys, torch, numpy as np, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
print(f"Python {sys.version}\nPyTorch {torch.__version__}  CUDA {torch.version.cuda}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. Configuration

All defaults live in `configs/defaults.py`. You can override via CLI or modify `Stage1Config` directly.

In [ ]:
from configs import PATHS, STAGE1_DEFAULT_CONFIG as C
from configs.defaults import get_device, linear_ramp
from training.gpu_setup import detect_gpu_setup, patch_autocast_for_bf16

gpu_info = detect_gpu_setup()
print(f"GPU: {gpu_info}\n")
print(f"Config:\n  epochs={C.epochs}  lr={C.lr}  batch_size={C.batch_size}")
print(f"  image_size={C.image_size}  warmup_epochs={C.warmup_epochs}")
print(f"  use_context_encoder={C.use_context_encoder}  use_stain_aug={C.use_stain_aug}")
print(f"  fine_tune_last_n_blocks={C.fine_tune_last_n_blocks}")
print(f"  Data: {PATHS.data_dir}")

---
## 2. Data Constants & Dataset

In [ ]:
from data.constants import (
    INTERNAL_TISSUE_ID_TO_NAME, NUM_TISSUE_CLASSES,
    PUMA_NUCLEI_ID_TO_NAME, NUM_NUCLEI_CLASSES,
    TISSUE_CLASS_WEIGHTS, NUCLEI_CLASS_WEIGHTS,
    RARE_TISSUE_IDS, RARE_NUCLEI_IDS,
)

print(f"Tissue classes ({NUM_TISSUE_CLASSES}): {dict(INTERNAL_TISSUE_ID_TO_NAME)}")
print(f"Nuclei classes  ({NUM_NUCLEI_CLASSES}): {dict(PUMA_NUCLEI_ID_TO_NAME)}")
print(f"Rare tissue IDs: {RARE_TISSUE_IDS}")
print(f"Rare nuclei IDs: {RARE_NUCLEI_IDS}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar(range(len(TISSUE_CLASS_WEIGHTS)), TISSUE_CLASS_WEIGHTS, color="tab:blue")
axes[0].set_title("Tissue class weights")
axes[1].bar(range(len(NUCLEI_CLASS_WEIGHTS)), NUCLEI_CLASS_WEIGHTS, color="tab:orange")
axes[1].set_title("Nuclei class weights")
plt.tight_layout(); plt.show()

### Dataset instantiation + transforms

In [ ]:
from data.dataset import PUMADataset, get_train_transforms, get_val_transforms
from data.dataset.transforms import TrainTransform

# Quick demo: instantiate dataset and inspect a sample
val_tfms = get_val_transforms(C.image_size)
train_tfms = get_train_transforms(C.image_size, use_stain_aug=C.use_stain_aug)

ds = PUMADataset(
    data_dir=str(PATHS.data_dir),
    transforms=train_tfms,
    use_context=C.use_context_encoder,
)
print(f"Dataset size: {len(ds)}")
sample = ds[0]
print(f"Keys: {list(sample.keys())}")
print(f"Image: {sample['image'].shape}  tissue: {sample['tissue_sem'].shape}")
print(f"nuclei_np: {sample['nuclei_np'].shape}  nc: {sample['nuclei_nc'].shape}  hv: {sample['nuclei_hv'].shape}")
print(f"site_id: {sample['site_id']}  base_name: {sample['base_name']}")
if C.use_context_encoder:
    print(f"context_roi: {sample.get('context_roi', 'N/A')}")

In [ ]:
# Visualise a few training samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
inv_mean = [-m/s for m,s in zip([0.485,0.456,0.406], [0.229,0.224,0.225])]
inv_std  = [1/s for s in [0.229,0.224,0.225]]
for i in range(2):
    s = ds[i]
    img = s["image"].permute(1,2,0).numpy()
    img = img * inv_std + inv_mean
    axes[i,0].imshow(np.clip(img, 0, 1))
    axes[i,0].set_title(f"Image {i}")
    axes[i,1].imshow(s["tissue_sem"], cmap="tab10", vmin=0, vmax=4)
    axes[i,1].set_title("Tissue")
    axes[i,2].imshow(s["nuclei_nc"], cmap="tab10", vmin=0, vmax=9)
    axes[i,2].set_title("Nuclei class")
    axes[i,3].imshow(s["nuclei_hv"][0], cmap="RdBu_r")
    axes[i,3].set_title("HV-X")
plt.tight_layout(); plt.show()

---
## 3. Model Architecture

In [ ]:
from models import UnifiedPanopticNet, build_cnn_backbone
from models.encoder import UnifiedPanopticEncoder
from models.decoders import (
    ParallelDecoders, MutualFeatureExchange,
    DeepLabV3PlusTissueHead, CellViTPlusPlusNucleiDecoder, ASPP,
)
from models.fpn_aggregator import HierarchicalFPN
from models.components import ContextEncoder, ContextFusionModule, BoundaryAttentionModule

# Build CNN backbone (ConvNeXt-Tiny, no pretrained weights needed for demo)
cnn = build_cnn_backbone(pretrained=False)
print(f"CNN backbone: ConvNeXt-Tiny  feature_dims={cnn.feature_info.channels()}")

# Full model (with placeholder encoder — won't load Virchow2 in this demo)
model = UnifiedPanopticNet(
    cnn_model=cnn,
    load_encoder_weights=False,
    use_context_encoder=C.use_context_encoder,
)
print(f"Model param count: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
# Forward pass on dummy data
B = 2
dummy_img = torch.randn(B, 3, C.image_size, C.image_size)
dummy_ctx = torch.randn(B, 3, C.context_roi_size, C.context_roi_size) if C.use_context_encoder else None
with torch.no_grad():
    out = model(dummy_img, context_roi=dummy_ctx)
for k, v in out.items():
    print(f"{k:12s}: {list(v.shape)}")

### Individual component inspection

In [ ]:
fpn = HierarchicalFPN(vit_dim=1280, cnn_dims=[96,192,384,768], fpn_dim=256)
vit_tokens = torch.randn(1, 257, 1280)
cnn_feats = [torch.randn(1, d, 256//(2**i), 256//(2**i)) for i, d in enumerate([96,192,384,768])]
vit_inter = torch.randn(4, 1, 1280, 64, 64)
fpn_out, low_feat = fpn(vit_tokens, cnn_feats, vit_inter, img_size=256)
print(f"FPN outputs: { {k: list(v.shape) for k,v in fpn_out.items()} }")
print(f"Low-level feat: {list(low_feat.shape)}")

In [ ]:
decoders = ParallelDecoders(fpn_dim=256, num_tissue=5, num_nuclei=10, low_level_channels=96)
with torch.no_grad():
    tissue, np, nc, hv, boundary = decoders(fpn_out, low_feat, vit_inter)
print(f"tissue: {list(tissue.shape)}   np: {list(np.shape)}   nc: {list(nc.shape)}")
print(f"hv: {list(hv.shape)}   boundary: {list(boundary.shape)}")

In [ ]:
# Context encoder + fusion demo
if C.use_context_encoder:
    ctx_enc = ContextEncoder(output_dim=256, output_mode="global")
    ctx_fus = ContextFusionModule(context_dim=256, fpn_dim=256)
    dummy_ctx = torch.randn(1, 3, C.context_roi_size, C.context_roi_size)
    ctx_desc = ctx_enc(dummy_ctx)
    print(f"Context descriptor: {list(ctx_desc.shape)}")
    fused = ctx_fus(fpn_out, ctx_desc)
    print(f"Fused FPN p1: {list(fused['p1'].shape)}")

---
## 4. Loss & Metrics

In [ ]:
from utils import MultiTaskUncertaintyLoss, PUMAMetrics

criterion = MultiTaskUncertaintyLoss(
    num_tissue_classes=5,
    num_nuclei_classes=10,
    tissue_class_weights=torch.tensor(TISSUE_CLASS_WEIGHTS, dtype=torch.float32),
    nuclei_class_weights=torch.tensor(NUCLEI_CLASS_WEIGHTS, dtype=torch.float32),
)
metrics = PUMAMetrics(num_tissue=5, num_nuclei=10)

# Demo forward+loss on dummy batch
loss, branch_losses = criterion(out, {
    "tissue_sem": torch.randint(0, 5, (B, C.image_size, C.image_size)),
    "nuclei_np": torch.randint(0, 2, (B, C.image_size, C.image_size)),
    "nuclei_nc": torch.randint(0, 10, (B, C.image_size, C.image_size)),
    "nuclei_hv": torch.randn(B, 2, C.image_size, C.image_size),
})
print(f"Total loss: {loss.item():.4f}")
print(f"Branch losses: tissue={branch_losses[0]:.4f}  np={branch_losses[1]:.4f}  nc={branch_losses[2]:.4f}  hv={branch_losses[3]:.4f}")

In [ ]:
# Metrics demo
val_metrics = metrics.calculate_all_metrics(out, {
    "tissue_sem": torch.randint(0, 5, (B, C.image_size, C.image_size)),
    "nuclei_np": torch.randint(0, 2, (B, C.image_size, C.image_size)),
    "nuclei_nc": torch.randint(0, 10, (B, C.image_size, C.image_size)),
    "nuclei_hv": torch.randn(B, 2, C.image_size, C.image_size),
})
for k, v in val_metrics.items():
    if isinstance(v, float):
        print(f"  {k:25s}: {v:.4f}")

---
## 5. Scheduler & Schedule Visualization

In [ ]:
from utils.scheduler_utils import build_warmup_cosine_scheduler

optimizer = torch.optim.AdamW(model.parameters(), lr=C.lr, weight_decay=C.weight_decay)
scheduler = build_warmup_cosine_scheduler(optimizer, warmup_epochs=C.warmup_epochs, total_epochs=C.epochs)

# Simulate LR schedule
lrs = []
for ep in range(1, C.epochs + 1):
    for _ in range(10):  # simulate steps per epoch
        lrs.append(optimizer.param_groups[0]["lr"])
        scheduler.step()

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.axvline(C.warmup_epochs * 10, color="r", ls=":", label=f"warmup end (ep {C.warmup_epochs})")
plt.xlabel("Step"); plt.ylabel("LR"); plt.title("Warm-up Cosine Schedule"); plt.legend(); plt.show()

# Reset scheduler
scheduler = build_warmup_cosine_scheduler(optimizer, warmup_epochs=C.warmup_epochs, total_epochs=C.epochs)

In [ ]:
# Focal + SC-DFA ramp schedule
epochs = C.epochs
focal_w = [linear_ramp(e, C.focal_start_epoch, C.focal_full_epoch, C.focal_max_weight) for e in range(epochs)]
scdfa_w = [linear_ramp(e, C.sc_dfa_start_epoch, C.sc_dfa_full_epoch, C.sc_dfa_max_weight) for e in range(epochs)]

plt.figure(figsize=(8, 3))
plt.plot(focal_w, label="Focal-Tversky weight", lw=2)
plt.plot(scdfa_w, label="SC-DFA lambda", lw=2)
plt.axvline(C.focal_start_epoch, color="g", ls=":")
plt.axvline(C.sc_dfa_start_epoch, color="m", ls=":")
plt.xlabel("Epoch"); plt.ylabel("Weight"); plt.title("Smooth Schedule"); plt.legend(); plt.show()

---
## 6. Training Loop (Simulated)

Shows loss curves and metric tracking without running a full training.

In [ ]:
import math
np.random.seed(42)

num_epochs = 30
history = {
    "train_loss": [], "val_loss": [],
    "tissue_dice": [], "nuclei_dice": [], "rare_tissue_dice": [], "rare_nuclei_dice": [],
}

for ep in range(1, num_epochs + 1):
    decay = math.exp(-0.05 * ep)
    noise = 0.02 * np.random.randn()
    base = 2.5 * decay + 0.1 + noise
    history["train_loss"].append(base + 0.05 * np.random.randn())
    history["val_loss"].append(base + 0.03 * np.random.randn())
    progress = min(ep / 15, 1.0)
    history["tissue_dice"].append(0.6 * progress + 0.3 + 0.02 * np.random.randn())
    history["nuclei_dice"].append(0.4 * progress + 0.2 + 0.02 * np.random.randn())
    history["rare_tissue_dice"].append(0.3 * progress + 0.1 + 0.03 * np.random.randn())
    history["rare_nuclei_dice"].append(0.2 * progress + 0.05 + 0.03 * np.random.randn())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].set_title("Loss")
for k in ["tissue_dice", "nuclei_dice", "rare_tissue_dice", "rare_nuclei_dice"]:
    axes[1].plot(history[k], label=k)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice"); axes[1].legend(fontsize=8); axes[1].set_title("Validation Dice")
plt.tight_layout(); plt.show()

---
## 7. Trainer & CLI Entry Point

The actual training script is `training/stage1_trainer.py`. To train:

```bash
# Minimal run
python scripts/run_stage1.py --batch-size 1 --use-stain-aug --use-context-encoder

# Full training with custom params
python scripts/run_stage1.py --epochs 50 --lr 1e-4 --batch-size 1 --use-stain-aug
```

The trainer will:
- Load/save train/val split (`make_or_load_group_split`)
- Build data loaders with weighted sampling for rare classes
- Apply smooth schedule (Focal-Tversky ramp + SC-DFA ramp)
- Log metrics via `PUMAMetrics`
- Save best checkpoint by selection score

In [ ]:
from training import train_one_epoch, validate, extract_state_dict, safe_torch_save_entity
from training.stage1_trainer import apply_smooth_schedule, save_checkpoint
from training.gpu_setup import cleanup_gpu_cache

# Check that train_one_epoch and validate are importable
print(f"train_one_epoch: {train_one_epoch.__name__}")
print(f"validate: {validate.__name__}")
print(f"apply_smooth_schedule: {apply_smooth_schedule.__name__}")
print(f"save_checkpoint: {save_checkpoint.__name__}")
print("\nAll training primitives import OK.")

---
## 8. Inference Demo (TTA + Post-processing)

In [ ]:
from inference.infer_wsi import apply_tta, parse_args as infer_parse_args
from inference.model_loader import load_stage1_checkpoint

# TTA demo on dummy data
class DummyModel(torch.nn.Module):
    def forward(self, x, site_ids=None, context_roi=None):
        return {
            "tissue": torch.randn(1, 5, *x.shape[-2:]),
            "np": torch.randn(1, 1, *x.shape[-2:]),
            "nc": torch.randn(1, 10, *x.shape[-2:]),
            "hv": torch.randn(1, 2, *x.shape[-2:]),
            "boundary": torch.randn(1, 1, *x.shape[-2:]),
        }

dummy = DummyModel()
tensor = torch.randn(1, 3, 256, 256)
out_tta = apply_tta(dummy, tensor, site_ids=None, use_tta=True)
print(f"TTA outputs: { {k: list(v.shape) for k,v in out_tta.items()} }")

In [ ]:
# Inference command (for reference)
print("Inference usage:")
print("  python scripts/run_inference.py --input /path/to/tif --output /path/out")
print("  python scripts/run_inference.py --input /path/wsidata --use-tta")

---
## 9. Verify All Exports

In [ ]:
from models import UnifiedPanopticNet, build_cnn_backbone
from models.encoder import UnifiedPanopticEncoder
from models.decoders import (
    ParallelDecoders, MutualFeatureExchange,
    DeepLabV3PlusTissueHead, CellViTPlusPlusNucleiDecoder,
    ASPP, HoVerNeXtNucleiHead,
)
from models.fpn_aggregator import HierarchicalFPN
from models.components import ContextEncoder, ContextFusionModule, BoundaryAttentionModule
from data.constants import *
from data.dataset import PUMADataset, get_train_transforms, get_val_transforms
from training import (
    train_one_epoch, validate, extract_state_dict, safe_torch_save,
    detect_gpu_setup, cleanup_gpu_cache, logger,
)
from utils import MultiTaskUncertaintyLoss, PUMAMetrics, SCDFA, build_warmup_cosine_scheduler
from configs import PATHS, STAGE1_DEFAULT_CONFIG, INFERENCE_DEFAULT_CONFIG
print("All v8 imports OK.")

In [ ]:
cleanup_gpu_cache()
print("Done.")